# Validation


In [ ]:
import rasterio
import numpy as np

from pathlib import Path
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch


from rasterio.warp import reproject, Resampling



base_path = '/data/LEON/P6_forest_agri_uganda/'

## Load

In [ ]:
# Tree height files
treeheight_dir = Path(base_path + "predictions")
years = [2020, 2021, 2022, 2023, 2024]

treeheight_data = {}
for year in years:
    treeheight_filepath = treeheight_dir / f"bugoma_treeheight_{year}_v10_cog.tif"
    with rasterio.open(treeheight_filepath) as src:
        treeheight_data[year] = src.read(1)  # Read first band
        print(f"Loaded {year}: shape {treeheight_data[year].shape}, CRS: {src.crs}")

# Forest extent file
forest_extent_path = Path(base_path + "area/forest_extent_bugoma_2020.tif")
with rasterio.open(forest_extent_path) as src:
    forest_extent = src.read(1)
    print(f"Loaded forest extent: shape {forest_extent.shape}, CRS: {src.crs}")

print("\nAll files loaded successfully!")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Tree height
im1 = ax1.imshow(treeheight_data[2020], cmap='viridis')
ax1.set_title('Tree Height 2020')
ax1.axis('off')
plt.colorbar(im1, ax=ax1, label='Height (m)')

# Forest extent binary
forest_binary = forest_extent > 0
im2 = ax2.imshow(forest_binary, cmap='Greens')
ax2.set_title(f'Forest Extent ({forest_binary.sum():,} forest pixels)')
ax2.axis('off')

plt.tight_layout()
plt.show()

## Forest Baseline

In [ ]:
from rasterio.warp import reproject, Resampling

# Resample forest extent to match tree height grid
with rasterio.open(treeheight_filepath) as src_th:
    with rasterio.open(forest_extent_path) as src_fe:
        forest_extent_resampled = np.empty(treeheight_data[2020].shape, dtype=src_fe.dtypes[0])
        
        reproject(
            source=rasterio.band(src_fe, 1),
            destination=forest_extent_resampled,
            src_transform=src_fe.transform,
            src_crs=src_fe.crs,
            dst_transform=src_th.transform,
            dst_crs=src_th.crs,
            resampling=Resampling.nearest
        )

# Create baseline
forest_baseline = (forest_extent_resampled > 0) & (treeheight_data[2020] > 5)

print(f"Forest baseline pixels: {np.sum(forest_baseline):,}")

In [ ]:
plt.imshow(forest_baseline, cmap='Greens')

## Deforestation progression from tree height

In [ ]:
# Pixels to ignore everywhere (corners)

INVALID_VAL = 7.806812
TOL = 1e-6
invalid_mask = np.isclose(treeheight_data[2020], INVALID_VAL, atol=TOL)
plt.imshow(invalid_mask)

In [ ]:

# --- Baseline forest mask (ignore invalid corners) --- #
forest_baseline = (
    (forest_extent_resampled > 0) &
    (~invalid_mask)
)

# --- Initialise deforestation map --- #
# 0 = non-forest
# 1 = stable forest
# 20–24 = year of deforestation
deforestation_map = np.zeros_like(treeheight_data[2020], dtype=int)
deforestation_map[forest_baseline] = 1

# --- Detect deforestation year by year --- #
defo_treeheight = 8
years = [2021, 2022, 2023, 2024]

for year in years:
    invalid_mask_year = np.isclose(treeheight_data[year], INVALID_VAL, atol=TOL)
    valid = ~invalid_mask_year

    still_forest = (deforestation_map == 1)
    height_dropped = (treeheight_data[year] < defo_treeheight) & valid

    newly_deforested = still_forest & height_dropped
    deforestation_map[newly_deforested] = year - 2000

    print(f"{year}: {np.sum(newly_deforested):,} pixels deforested")

# --- 5. Summary ---
print("\nStable non-forest (0):", np.sum(deforestation_map == 0))
print("Stable forest (1):", np.sum(deforestation_map == 1))
print("Total deforested:",
      np.sum((deforestation_map >= 20) & (deforestation_map <= 24)))


In [ ]:
# Initialize deforestation map
# 0 = non-forest, 1 = stable forest, 20-24 = year of deforestation

defo_treeheight = 6

deforestation_map = np.zeros_like(treeheight_data[2020], dtype=int)

# Set initial values
deforestation_map[~forest_baseline] = 0  # Non-forest areas
deforestation_map[forest_baseline] = 1   # Initial forest

# Track deforestation year by year
years = [2021, 2022, 2023, 2024]

for year in years:
    # Find pixels that are still forest (value = 1) but height dropped below defo_treeheight (treeheight bias)
    still_forest = (deforestation_map == 1)
    height_dropped = (treeheight_data[year] < defo_treeheight)
    newly_deforested = still_forest & height_dropped
    
    # Mark with deforestation year (20, 21, 22, 23, 24)
    deforestation_map[newly_deforested] = year - 2000
    
    print(f"{year}: {np.sum(newly_deforested):,} pixels deforested")

# Summary
print(f"\nStable non-forest (0): {np.sum(deforestation_map == 0):,}")
print(f"Stable forest (1): {np.sum(deforestation_map == 1):,}")
print(f"Total deforested: {np.sum((deforestation_map >= 20) & (deforestation_map <= 24)):,}")

In [ ]:
# Create colormap
plt.figure(figsize=(12, 8))

# Custom colors: gray (non-forest), green (forest), red gradient (deforestation years)
cmap = plt.cm.colors.ListedColormap(['gray', 'darkgreen', 'orange', 'red', 'darkred', 'black', 'purple'])
bounds = [0, 1, 21, 22, 23, 24, 25]
norm = plt.cm.colors.BoundaryNorm(bounds, cmap.N)

im = plt.imshow(deforestation_map, cmap=cmap, norm=norm)
plt.colorbar(im, label='0=non-forest, 1=forest, 20-24=deforestation year', ticks=[0, 1, 21, 22, 23, 24])
plt.title('Forest Change Detection 2019-2024')
plt.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Export with LZW compression
output_dir = Path(base_path + "tess_deforestation")
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "deforestation_map_tess_2020_2024_6m.tif"

with rasterio.open(treeheight_filepath) as src:
    with rasterio.open(output_path, 'w', driver='GTiff', height=src.height, 
                       width=src.width, count=1, dtype='int16', crs=src.crs, 
                       transform=src.transform, compress='lzw') as dst:
        dst.write(deforestation_map, 1)

print(f"Saved: {output_path}")

# Assess S1 deforestation tracker to predicted tree height and Hansen

In [ ]:
# ---------------- PATHS ----------------
s1_path = Path(base_path) / "s1_deforested/s1_deforestation_mask_20-24.tif"
th_path = Path(base_path) / "tess_deforestation/deforestation_map_tess_2020_2024_6m.tif"
hansen_path = Path(base_path) / "hansen/hansen_bugoma_32636.tif"


# ----------- LOAD S1 (REFERENCE GRID) -----------
with rasterio.open(s1_path) as src:
    s1 = src.read(1).astype(bool)
    dst_transform = src.transform
    dst_crs = src.crs
    H, W = src.height, src.width

print("S1 deforestation pixels:", s1.sum())


# ----------- REPROJECT INVALID MASK -----------
invalid_mask_s1 = np.zeros((H, W), dtype=np.uint8)

with rasterio.open(th_path) as src:
    reproject(
        source=invalid_mask.astype(np.uint8),   # <-- invalid mask variable
        destination=invalid_mask_s1,
        src_transform=src.transform,
        src_crs=src.crs,
        dst_transform=dst_transform,
        dst_crs=dst_crs,
        resampling=Resampling.nearest
    )

invalid_mask_s1 = invalid_mask_s1.astype(bool)


# ----------- LOAD & REPROJECT TREEHEIGHT -----------
th_arr = np.zeros((H, W), dtype=np.uint8)

with rasterio.open(th_path) as src:
    reproject(
        source=rasterio.band(src, 1),
        destination=th_arr,
        src_transform=src.transform,
        src_crs=src.crs,
        dst_transform=dst_transform,
        dst_crs=dst_crs,
        resampling=Resampling.nearest
    )

treeheight_defor = ((th_arr >= 20) & (th_arr <= 24))
print("Treeheight deforestation pixels:", treeheight_defor.sum())


# ----------- LOAD & REPROJECT HANSEN -----------
h_arr = np.zeros((H, W), dtype=np.uint8)

with rasterio.open(hansen_path) as src:
    reproject(
        source=rasterio.band(src, 1),
        destination=h_arr,
        src_transform=src.transform,
        src_crs=src.crs,
        dst_transform=dst_transform,
        dst_crs=dst_crs,
        resampling=Resampling.nearest
    )

hansen_defor = ((h_arr >= 20) & (h_arr <= 24))
print("Hansen deforestation pixels:", hansen_defor.sum())


# ----------- APPLY VALID MASK -----------
valid_mask = ~invalid_mask_s1

s1_valid         = s1               & valid_mask
hansen_valid     = hansen_defor     & valid_mask
treeheight_valid = treeheight_defor & valid_mask


# ---------------- METRICS FUNCTION ----------------
def compute_metrics(pred, ref):
    tp = int(np.sum(pred & ref))
    fp = int(np.sum(pred & ~ref))
    fn = int(np.sum(~pred & ref))
    tn = int(np.sum(~pred & ~ref))
    precision = tp/(tp+fp)*100 if (tp+fp)>0 else 0
    recall    = tp/(tp+fn)*100 if (tp+fn)>0 else 0
    f1 = (2*precision*recall)/(precision+recall) if (precision+recall)>0 else 0
    return tp, fp, fn, tn, precision, recall, f1


# ---------------- COMPUTE BOTH COMPARISONS ----------------
tp_h, fp_h, fn_h, tn_h, p_h, r_h, f1_h = compute_metrics(s1_valid, hansen_valid)
tp_t, fp_t, fn_t, tn_t, p_t, r_t, f1_t = compute_metrics(s1_valid, treeheight_valid)


# ---------------- PRINT METRICS ----------------
print("\n==== S1 vs Hansen (valid only) ====")
print("TP:", tp_h, " FP:", fp_h, " FN:", fn_h, " TN:", tn_h)
print(f"Precision: {p_h:.1f}%  Recall: {r_h:.1f}%  F1: {f1_h:.1f}%")

print("\n==== S1 vs Treeheight (valid only) ====")
print("TP:", tp_t, " FP:", fp_t, " FN:", fn_t, " TN:", tn_t)
print(f"Precision: {p_t:.1f}%  Recall: {r_t:.1f}%  F1: {f1_t:.1f}%")

In [ ]:
# ---------------- VISUALIZATION ----------------
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# ----------- ROW 1: raw masked layers -----------
axes[0,0].imshow(s1_valid, cmap="Reds")
axes[0,0].set_title(f"S1 deforestation (valid)\n{s1_valid.sum():,} px")
axes[0,0].axis("off")

axes[0,1].imshow(hansen_valid, cmap="Blues")
axes[0,1].set_title(f"Hansen (valid)\n{hansen_valid.sum():,} px")
axes[0,1].axis("off")

axes[0,2].imshow(treeheight_valid, cmap="Greens")
axes[0,2].set_title(f"Treeheight (valid)\n{treeheight_valid.sum():,} px")
axes[0,2].axis("off")


# ----------- AGREEMENT MAPS (masked) -----------
def make_agreement(pred, ref):
    out = np.zeros_like(pred, dtype=np.uint8)
    out[(~pred) & (~ref)] = 0
    out[(~pred) & (ref)]  = 1
    out[(pred)  & (~ref)] = 2
    out[(pred)  & (ref)]  = 3
    return out

agree_h = make_agreement(s1_valid, hansen_valid)
agree_t = make_agreement(s1_valid, treeheight_valid)

cmap = ListedColormap(["#d9d9d9","#fee391","#fb6a4a","#31a354"])

# S1 vs Hansen
axes[1,1].imshow(agree_h, cmap=cmap, vmin=0, vmax=3)
axes[1,1].set_title(f"S1 vs Hansen (valid)\nF1={f1_h:.1f}%")
axes[1,1].axis("off")

# S1 vs Treeheight
axes[1,2].imshow(agree_t, cmap=cmap, vmin=0, vmax=3)
axes[1,2].set_title(f"S1 vs Treeheight (valid)\nF1={f1_t:.1f}%")
axes[1,2].axis("off")


# Legend
legend = [
    Patch(color="#d9d9d9", label="TN: No loss"),
    Patch(color="#fee391", label="FN: Missed loss"),
    Patch(color="#fb6a4a", label="FP: False loss"),
    Patch(color="#31a354", label="TP: Agreement"),
]

axes[1,0].axis("off")
axes[1,0].legend(handles=legend, loc="center", fontsize=10)
axes[1,0].set_title("Legend")

plt.tight_layout()
plt.show()